# 09 — Toolsets & MCP

**Scope** — what the agent can touch.

When you finish you can:

- bundle tools into a `FunctionToolset` and give it to more than one agent,
- fix a tool-name clash with `.prefixed`,
- hide tools with `.filtered`, always or depending on who is asking,
- plug in an MCP server with `MCPToolset` and scope it the same way.

> This notebook runs **keyless**. `TestModel` and `FunctionModel` (from 08) stand in for the model,
> so you need no API key.

In [ ]:
from learningdantic import setup

MODEL = setup()

## A toolset is a bundle of tools

In 04 you put tools straight onto one agent with `@agent.tool_plain`. A **toolset** is a bundle of
tools that lives on its own. You write it once and pass it to any agent with `toolsets=[...]`.
`@kitchen.tool_plain` works just like `@agent.tool_plain`.

The toolsets you pass are the agent's **scope**: the model can only call tools it is shown. To prove
what it was shown, read `last_model_request_parameters.function_tools` on a `TestModel`. The small
`offered()` helper below does that, and we use it all through this notebook.

In [ ]:
from pydantic_ai import Agent, FunctionToolset
from pydantic_ai.models.test import TestModel


def offered(test_model):
    """Names of the tools the model was shown on its last request."""
    return [tool.name for tool in test_model.last_model_request_parameters.function_tools]


kitchen = FunctionToolset()


@kitchen.tool_plain
def lookup(ingredient: str) -> str:
    """Grams per cup of a baking ingredient."""
    return {"flour": "120 g per cup", "sugar": "200 g per cup"}.get(ingredient.lower(), "unknown")


@kitchen.tool_plain
def scale(amount: float, factor: float) -> float:
    """Multiply an amount by a factor (2 doubles a recipe, 0.5 halves it)."""
    return amount * factor


baker_model, pastry_model = TestModel(), TestModel()
baker = Agent(baker_model, toolsets=[kitchen])
pastry_chef = Agent(pastry_model, toolsets=[kitchen])  # the same toolset, a second agent

await baker.run("Double 2 cups of flour.")
await pastry_chef.run("How many grams is a cup of sugar?")
print("baker sees:      ", offered(baker_model))
print("pastry chef sees:", offered(pastry_model))

> ✅ **Checkpoint** — both lines print `['lookup', 'scale']`. One toolset, two agents: fix a tool
> once and both agents get the fix.

## Two toolsets, one name: `.prefixed`

Tool names must be unique inside one agent. Below, a `shop` toolset also has a tool called `lookup`.
Pydantic AI will not guess which one you meant, so the run raises `UserError`. It happens when the
agent runs, not when you build it.

The fix is `.prefixed("shop")`. It returns a wrapped copy whose tool names start with `shop_`. The
original toolset does not change.

In [ ]:
from pydantic_ai import UserError

shop = FunctionToolset()


@shop.tool_plain  # same name as kitchen's tool, on purpose (noqa tells the linter)
def lookup(item: str) -> str:  # noqa: F811
    """Price of one bag of a supply item."""
    return {"flour": "$2 a bag", "sugar": "$3 a bag"}.get(item.lower(), "not stocked")


@shop.tool_plain
def order(item: str, bags: int) -> str:
    """Order bags of a supply item. This spends money."""
    return f"ordered {bags} bag(s) of {item}"


try:
    await Agent(TestModel(), toolsets=[kitchen, shop]).run("What does flour cost?")
except UserError as error:
    print("clash    ->", type(error).__name__)

model = TestModel()
both = [kitchen.prefixed("kitchen"), shop.prefixed("shop")]
await Agent(model, toolsets=both).run("What does flour cost?")
print("prefixed ->", offered(model))

> ✅ **Checkpoint** — `clash    -> UserError`, then
> `prefixed -> ['kitchen_lookup', 'kitchen_scale', 'shop_lookup', 'shop_order']`.

## Hide tools with `.filtered`

This course hides tools one way: `.filtered(...)`. You give it a small function that gets the run
context and one tool's definition, and returns `True` to keep the tool or `False` to hide it.

`lambda ctx, td: ...` is a one-line function with no name. `td.name` is the tool's name.

A hidden tool is not refused, it is **gone**: the model is never told it exists, so it cannot call
it. Below, the helper may check prices but never order.

In [ ]:
browse_only = shop.filtered(lambda ctx, td: td.name != "order")

model = TestModel()
await Agent(model, toolsets=[browse_only]).run("Order 5 bags of flour.")
print("browse only ->", offered(model))

> ✅ **Checkpoint** — `browse only -> ['lookup']`. The prompt asked for an order, but `order` was not
> on the list the model saw.

## Filter by who is asking

The filter's `ctx` is the same `RunContext` you met in 05, so it can read `ctx.deps`. Here the deps
say who is using the agent, and only a manager sees `order`. The filter runs again on every request,
so one agent can show different tools to different people.

`@dataclass` turns a class with typed fields into a simple data holder (as in 05).
`TestModel(call_tools=[])` calls no tools: we only want to see the list.

In [ ]:
from dataclasses import dataclass


@dataclass
class Staff:
    role: str  # "cook" or "manager"


model = TestModel(call_tools=[])
helper = Agent(
    model,
    deps_type=Staff,
    toolsets=[shop.filtered(lambda ctx, td: td.name != "order" or ctx.deps.role == "manager")],
)

for role in ["cook", "manager"]:
    await helper.run("Order 5 bags of flour.", deps=Staff(role))
    print(f"{role:>7} sees:", offered(model))

> ✅ **Checkpoint** — `cook sees: ['lookup']` and `manager sees: ['lookup', 'order']`. Same agent, same
> prompt; the deps decided the scope.

## An MCP server is just another toolset

**MCP** (Model Context Protocol) is a standard plug for tools. The tools live in a **server**: usually
another program or another machine, often written by someone else. Your agent connects with
`MCPToolset(...)`, and from then on it is just another toolset. Everything above works on it.

To stay keyless and offline, this server lives inside this Python process. `MCPServer` comes from
the `mcp` package, which is already installed. `@pantry.tool()` registers a tool, and its docstring
becomes the description, as in 04.

`ToolError` is the server's version of `ModelRetry` from 04: the model sees your message and can try
again. Any other exception shows the model only "Error executing tool". `{ingredient!r}` inside an
f-string prints the value with quotes, so `'flor'` shows up clearly in the error.

Real servers run somewhere else: you give `MCPToolset` a URL or a script path instead of a server
object. See the [MCP client docs](https://pydantic.dev/docs/ai/mcp/client).

In [ ]:
from mcp.server.mcpserver import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic_ai.mcp import MCPToolset

pantry = MCPServer(name="pantry", log_level="WARNING")  # WARNING hides the server's routine logs
STOCK = {"flour": 4, "sugar": 1, "butter": 2}


@pantry.tool()
def check_stock(ingredient: str) -> int:
    """Bags of an ingredient left in the pantry. Known: flour, sugar, butter."""
    if ingredient.lower() not in STOCK:
        raise ToolError(f"Unknown ingredient {ingredient!r}. Use one of: {', '.join(STOCK)}.")
    return STOCK[ingredient.lower()]


@pantry.tool()
def throw_out(ingredient: str) -> str:
    """Throw away everything left of an ingredient."""
    STOCK[ingredient.lower()] = 0
    return f"threw out the {ingredient}"


pantry_tools = MCPToolset(pantry)  # in-process: no subprocess, no network, no key

model = TestModel(call_tools=[])
await Agent(model, toolsets=[pantry_tools]).run("How much flour is left?")
print("pantry offers ->", offered(model))

> ✅ **Checkpoint** — `pantry offers -> ['check_stock', 'throw_out']`. The agent connected to the
> server, asked it for its tools and showed them to the model. Each run opens and closes that
> connection by itself.

## Scope the server with `.filtered`

You can't edit someone else's server, but you decide which of its tools your agent sees. Same
`.filtered`, same kind of lambda.

In [ ]:
safe_pantry = pantry_tools.filtered(lambda ctx, td: td.name != "throw_out")

model = TestModel(call_tools=[])
await Agent(model, toolsets=[safe_pantry]).run("Throw out the sugar.")
print("safe pantry ->", offered(model))

> ✅ **Checkpoint** — `safe pantry -> ['check_stock']`. The server still has `throw_out`; your agent
> just can't see it.

## A server error becomes a retry

When a server tool raises `ToolError`, the agent sends the message back to the model as a retry: the
same loop as `ModelRetry` in 04. To watch the loop without a key, a `FunctionModel` plays a model that
misspells "flour", reads the error, and fixes its call.

In [ ]:
from pydantic_ai.messages import (
    ModelResponse,
    RetryPromptPart,
    TextPart,
    ToolCallPart,
    ToolReturnPart,
)
from pydantic_ai.models.function import FunctionModel


def typo_then_fix(messages, info):
    """A scripted model: misspell the ingredient, read the error, fix it, then answer."""
    last = messages[-1].parts
    if any(isinstance(part, RetryPromptPart) for part in last):
        return ModelResponse(parts=[ToolCallPart("check_stock", {"ingredient": "flour"})])
    if any(isinstance(part, ToolReturnPart) for part in last):
        return ModelResponse(parts=[TextPart(f"There are {last[0].content} bags of flour left.")])
    return ModelResponse(parts=[ToolCallPart("check_stock", {"ingredient": "flor"})])


result = await Agent(FunctionModel(typo_then_fix), toolsets=[safe_pantry]).run("How much flour?")
for message in result.all_messages():
    for part in message.parts:
        if isinstance(part, ToolCallPart):
            print("call  :", part.tool_name, part.args)
        elif isinstance(part, RetryPromptPart):
            print("retry :", part.content)
        elif isinstance(part, ToolReturnPart):
            print("return:", part.content)
print("answer:", result.output)

> ✅ **Checkpoint** — five lines: a call with `'flor'`, a `retry` line,
> `Error executing tool check_stock: Unknown ingredient 'flor'. Use one of: flour, sugar, butter.`
> (the server adds the `Error executing tool …:` prefix; the rest is your `ToolError` message), a call
> with `'flour'`, `return: 4`,
> and `answer: There are 4 bags of flour left.` The error message told the model how to fix its call,
> so a clear error message is a way to steer.

## 🏋️ Exercise — a read-only recipe server

Build an `MCPServer` called `recipes` with three tools: `list_recipes()`, `get_recipe(name)` and
`delete_recipe(name)`. `get_recipe` raises `ToolError` for an unknown name, and the message lists the
known names.

1. Connect it with `MCPToolset` so the agent can read recipes but **never sees** `delete_recipe`.
   Prove it with a `TestModel` and `offered()`.
2. Write a `FunctionModel` that asks for `"pancake"` (the recipe is called `"pancakes"`), gets the
   error, retries with the right name and answers with the recipe. Check that the answer contains
   the recipe text.

*Stretch (optional):* show `delete_recipe` only when `ctx.deps.role == "owner"`, and prove it for both
roles.

Solution: [`../solutions/09_scoped_recipe_server.ipynb`](../solutions/09_scoped_recipe_server.ipynb)

## Next →

[10 — Capabilities & guardrails](10_capabilities_and_guardrails.ipynb): pack a filtered toolset
together with the instructions that explain it, and add guardrails such as approval before a risky
tool runs.